# Lab type: review
# Course: ML203 — Unsupervised Learning & Clustering
# Lesson: Anomaly Detection
# Task: The code below is correct and working. Read each section, run it, then answer the judgment questions in the markdown cells below each block.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

## Step 1: Generate Sensor Data

We simulate a manufacturing line with five sensor readings (temperature, pressure, vibration, humidity, voltage). Normal operation follows a multivariate Gaussian. A small number of injected anomalies represent fault conditions.

In [ ]:
# 300 normal operating points — correlated sensor readings
mean = np.array([70, 5.0, 0.3, 45, 230])
cov = np.array([
    [4.0, 1.2, 0.1, 0.0, 0.5],
    [1.2, 0.4, 0.0, 0.0, 0.1],
    [0.1, 0.0, 0.01, 0.0, 0.0],
    [0.0, 0.0, 0.0, 9.0, 0.0],
    [0.5, 0.1, 0.0, 0.0, 25.0]
])
X_normal = rng.multivariate_normal(mean, cov, size=300)

# 15 anomalous points — sensor combinations outside normal operating range
X_anomalies = np.array([
    [85, 7.5, 0.9, 55, 260],   # overheating + high vibration
    [60, 3.0, 0.6, 40, 210],   # under-pressure + elevated vibration
    [90, 8.2, 1.1, 60, 270],
    [55, 2.5, 0.7, 38, 205],
    [88, 7.8, 0.8, 50, 265],
    [92, 8.5, 1.2, 62, 275],
    [58, 2.8, 0.5, 35, 200],
    [86, 7.2, 0.9, 52, 255],
    [63, 3.5, 0.6, 42, 215],
    [91, 8.0, 1.0, 58, 268],
    [57, 2.6, 0.8, 37, 202],
    [87, 7.6, 1.1, 53, 262],
    [62, 3.2, 0.5, 41, 212],
    [89, 8.1, 0.9, 59, 272],
    [56, 2.7, 0.7, 36, 203],
])

# Combine and record ground-truth labels for later evaluation
X = np.vstack([X_normal, X_anomalies])
y_true = np.array([0] * 300 + [1] * 15)  # 0 = normal, 1 = anomaly

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Dataset: {X.shape[0]} points, {X.shape[1]} features")
print(f"True anomaly rate: {y_true.mean()*100:.1f}%")

## Part 1: DBSCAN Noise Points

In [ ]:
# eps calibrated from the k-distance graph; min_samples chosen relative to dataset size
db = DBSCAN(eps=1.2, min_samples=8)
db_labels = db.fit_predict(X_scaled)

db_anomaly_pred = (db_labels == -1).astype(int)

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = db_anomaly_pred.sum()

print(f"Clusters found: {n_clusters}")
print(f"Noise points (anomaly candidates): {n_noise} ({n_noise/len(X)*100:.1f}%)")
print()
print(classification_report(y_true, db_anomaly_pred, target_names=["normal", "anomaly"]))

**Question 1:** DBSCAN found one tight cluster and flagged noise points as anomaly candidates. If you tightened `eps` to get a better-defined cluster boundary, more sparse-but-normal points would also become noise. How would you decide, in practice, whether a noise point is a genuine anomaly or just a legitimate low-density observation?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q1</summary>

**Treat noise points as candidates, not confirmed anomalies.** Apply a second-stage check: score each noise point using an independent method (e.g., an Isolation Forest score or z-score on individual sensor features) and flag it as anomalous only if both agree.

**Domain knowledge is the tie-breaker.** If your process has known valid low-density operating states — planned maintenance windows, start-up ramps, or sensor calibration periods — those will produce legitimate sparse observations that DBSCAN correctly excludes from the main cluster. Cross-reference noise timestamps against operational logs before escalating.

**Rule:** A noise point is an anomaly candidate. Confirm it with a second signal and domain context before acting on it.

</details>

**Question 2:** The precision and recall here are coupled to `eps` and `min_samples` — the same parameters that define your clusters. A colleague suggests running a grid search over `eps` to maximise recall for anomaly detection. What is wrong with this approach, and what would a better workflow look like?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q2</summary>

**What's wrong:** Optimising `eps` for maximum recall on a labelled evaluation set creates circular reasoning — you are tuning the density parameter (which defines what "normal" means) to match a specific anomaly list. This overfits the definition of normality to your evaluation labels. The resulting `eps` no longer reflects the genuine density structure of normal data.

**A better workflow:** Separate the concerns completely.
1. Calibrate `eps` on normal-only training data using the k-distance graph — no labels involved.
2. Evaluate detection performance at that fixed `eps` on a held-out set.
3. If recall is insufficient, do not adjust `eps`. Instead, add a complementary detector (Isolation Forest, reconstruction error) or threshold DBSCAN noise scores with a secondary model.

Adjusting the density parameter to maximise recall is tuning the clustering to be an anomaly detector — these are different objectives that require different tools.

</details>

## Part 2: Isolation Forest

In [ ]:
# Train on the full dataset; contamination set close to the true anomaly rate
iso = IsolationForest(contamination=0.05, random_state=42)
iso_preds = iso.fit_predict(X_scaled)  # 1 = inlier, -1 = anomaly
iso_scores = iso.decision_function(X_scaled)  # more negative = more anomalous

iso_anomaly_pred = (iso_preds == -1).astype(int)

print(f"Flagged as anomalies: {iso_anomaly_pred.sum()} ({iso_anomaly_pred.mean()*100:.1f}%)")
print()
print(classification_report(y_true, iso_anomaly_pred, target_names=["normal", "anomaly"]))

In [ ]:
# Score distribution: where normal and anomalous points sit on the decision boundary
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(iso_scores[y_true == 0], bins=30, alpha=0.6, label="Normal", color="steelblue")
ax.hist(iso_scores[y_true == 1], bins=10, alpha=0.7, label="Anomaly", color="tomato")
ax.axvline(0, color="black", linestyle="--", linewidth=1, label="Decision boundary (score=0)")
ax.set_xlabel("Anomaly score (more negative → more anomalous)")
ax.set_ylabel("Count")
ax.set_title("Isolation Forest: Score Distributions")
ax.legend()
plt.tight_layout()
plt.show()

**Question 3:** `contamination=0.05` was set because the true anomaly rate is approximately 4.8%. In a real deployment you rarely know the true rate. The score distribution plot above shows how cleanly the two groups separate. Given what you see, what would happen to precision and recall if you changed `contamination` to `0.10`? When would doubling the contamination estimate be the right business decision?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q3</summary>

**Effect of doubling contamination to 0.10:** The decision threshold is lowered, so more points are flagged as anomalous. Since the score distributions overlap only slightly (as the plot shows), recall increases — more genuine anomalies are captured — but precision falls because more normal-range points now cross the lower threshold.

**When this is the right business decision:** When the cost of a missed anomaly substantially outweighs the cost of a false alarm. For example, if each missed anomaly corresponds to a machine failure costing €10,000 and each false alarm costs €200 in unnecessary inspection, accepting extra false alarms is economically rational.

**How to calibrate:** Set `contamination` to reflect the actual cost ratio between missed anomalies and false alarms, not just the statistical anomaly rate. Use precision/recall curves across contamination values to make the tradeoff explicit.

</details>

**Question 4:** The code calls `fit_predict(X_scaled)` on the full dataset — normal and anomalous points together. The lesson notes that a train/test split is the correct approach for evaluation. In this specific lab, we already have ground-truth labels (`y_true`). Does fitting on the full dataset here introduce a valid concern? Under what real-world condition would it be a serious problem?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q4</summary>

**In this lab:** Mild optimism bias only. Because ground-truth labels exist and the goal is educational illustration, fitting on the full dataset doesn't undermine the demonstration — it slightly inflates the apparent separation between normal and anomalous score distributions.

**When it's a serious problem:** In production, when anomalies are frequent or systematic (e.g., a sensor that regularly malfunctions). If anomalous points are present in training data, the Isolation Forest learns to partially accommodate them as normal — it builds trees that require more splits to isolate them, raising the anomaly score threshold. The model is then partially trained to treat known anomalies as normal, guaranteeing reduced recall when similar anomalies appear at inference time.

**The correct production approach:** Fit only on confirmed-normal training data. If no clean training set is available, use a conservative `contamination` estimate and monitor score distributions after deployment.

</details>

## Part 3: PCA Reconstruction Error

In [ ]:
# Split before fitting the PCA so the reconstruction model never sees test anomalies
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_true, test_size=0.3, random_state=42, stratify=y_true
)

In [ ]:
# 3 components retain the dominant variance of the 5-feature normal distribution
pca = PCA(n_components=3)
pca.fit(X_train)

print(f"Variance explained by 3 components: {pca.explained_variance_ratio_.sum()*100:.1f}%")
print(f"Per-component: {pca.explained_variance_ratio_.round(3)}")

In [ ]:
# Reconstruction error: how much information is lost projecting to 3D and back
X_test_reconstructed = pca.inverse_transform(pca.transform(X_test))
reconstruction_error = np.mean((X_test - X_test_reconstructed) ** 2, axis=1)

# Threshold: flag the top 5% most poorly reconstructed points
threshold = np.percentile(reconstruction_error, 95)
pca_anomaly_pred = (reconstruction_error > threshold).astype(int)

print(f"Threshold (95th percentile): {threshold:.4f}")
print(f"Flagged as anomalies: {pca_anomaly_pred.sum()} ({pca_anomaly_pred.mean()*100:.1f}%)")
print()
print(classification_report(y_test, pca_anomaly_pred, target_names=["normal", "anomaly"]))

In [ ]:
# Reconstruction error distributions for normal vs anomalous test points
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(reconstruction_error[y_test == 0], bins=20, alpha=0.6, label="Normal", color="steelblue")
ax.hist(reconstruction_error[y_test == 1], bins=8, alpha=0.7, label="Anomaly", color="tomato")
ax.axvline(threshold, color="black", linestyle="--", linewidth=1, label=f"Threshold ({threshold:.3f})")
ax.set_xlabel("Mean squared reconstruction error")
ax.set_ylabel("Count")
ax.set_title("PCA: Reconstruction Error Distributions")
ax.legend()
plt.tight_layout()
plt.show()

**Question 5:** The threshold is set at the 95th percentile of reconstruction error on the test set. By construction, this flags roughly 5% of test points regardless of whether they are anomalies. Looking at the plot above: does this threshold appear well-calibrated for this dataset? What would you change, and why?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q5</summary>

**Is the 95th-percentile threshold well-calibrated?** If the reconstruction error plot shows a clear gap between the normal and anomalous distributions, the 95th-percentile threshold happens to fall in that gap — the result is good calibration by design for this synthetic dataset. If the distributions overlap, the percentile threshold flags many normal points regardless.

**The fundamental issue:** Thresholding at a fixed percentile *guarantees* flagging roughly 5% of points regardless of how many anomalies actually exist — it is a fixed-recall design, not a principled decision boundary.

**A better approach:** Set the threshold at the natural gap in the error distribution, or estimate the 99th percentile of reconstruction error on a held-out set of confirmed-normal observations and flag anything above that. This anchors the threshold to the behaviour of normal data, not an arbitrary quantile of the mixed test set.

</details>

**Question 6:** PCA was fit on `X_train` only. Why does it matter that the PCA is fit exclusively on training data? Describe specifically what would go wrong if you instead fit PCA on `X_scaled` (the full dataset including test anomalies) before splitting.

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q6</summary>

**Why X_train only:** The PCA must learn what "normal" looks like — the principal components should capture variance in normal operating conditions. If anomalous points are included during fitting, the components are rotated to partially explain anomaly variance, reducing reconstruction error for anomalies and making them harder to distinguish from normal points at test time.

**What goes wrong with PCA on the full dataset:** Anomalies are included in the covariance matrix, so the learned components accommodate their structure. At test time, anomalous points reconstruct better than they should — their reconstruction error is lower than it would be under a normal-only PCA — and the error distribution between normal and anomalous points overlaps more, reducing detection power.

**This is a data-leakage problem:** The reconstruction model no longer represents "what normal looks like"; it represents "what the full dataset, including anomalies, looks like."

</details>

## Part 4: Method Agreement

In [ ]:
# Re-run Isolation Forest on the same train/test split for a fair comparison
iso2 = IsolationForest(contamination=0.05, random_state=42)
iso2.fit(X_train)
iso2_preds = iso2.predict(X_test)
iso2_anomaly_pred = (iso2_preds == -1).astype(int)

# Agreement: both methods flag the same point
both_flag = (iso2_anomaly_pred == 1) & (pca_anomaly_pred == 1)
either_flag = (iso2_anomaly_pred == 1) | (pca_anomaly_pred == 1)

print(f"Flagged by Isolation Forest only: {((iso2_anomaly_pred==1) & (pca_anomaly_pred==0)).sum()}")
print(f"Flagged by PCA only:              {((iso2_anomaly_pred==0) & (pca_anomaly_pred==1)).sum()}")
print(f"Flagged by both (high confidence): {both_flag.sum()}")
print()
print("High-confidence anomalies (both methods agree):")
print(classification_report(y_test, both_flag.astype(int), target_names=["normal", "anomaly"]))

**Question 7:** Some test anomalies were flagged by only one method. Describe a plausible scenario — using the sensor data context — where PCA would flag a point that Isolation Forest misses. Then describe the reverse: a scenario where Isolation Forest flags a point that PCA misses.

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q7</summary>

**PCA flags, Isolation Forest misses:** A sensor reading where temperature (70°C) and pressure (5 bar) are each individually normal, but their simultaneous occurrence violates a known physical correlation — high temperature with high pressure together never occurs in normal operation. Isolation Forest partitions each feature independently in its random trees; it would not isolate a point whose individual feature values are within normal marginal ranges, even if the combination is impossible under normal conditions. PCA captures this because the correlated variance is encoded in principal components — a violation of the temperature-pressure correlation produces high reconstruction error.

**Isolation Forest flags, PCA misses:** A single sensor spike to 200°C when all other sensors read normally. If the first few principal components mostly capture shared variance across all five sensors (e.g., a global operating-level factor), the spike on one sensor is partially absorbed by the reconstruction because the other four sensors anchor the predicted value. Isolation Forest would isolate this point quickly — a single-feature extreme value requires very few splits to separate from the distribution.

</details>

**Question 8:** The "high confidence" set (both methods agree) has higher precision than either method alone. But requiring agreement also tends to lower recall. In the fraud detection context from the lesson — where missing a fraud is far more costly than a false alarm — would you use the agreement-only set, either method alone, or the union of both? Justify your answer in terms of the precision/recall tradeoff.

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q8</summary>

**For fraud detection: use the union of both methods.** Requiring agreement (high confidence set) maximises precision but lowers recall — more frauds slip through undetected, which is the most costly error when the asymmetry between missed fraud and false alarm is high.

**The union gives the highest recall** at the cost of more false alarms. In fraud detection, where each missed fraud may cost thousands and each false-alarm investigation costs tens, the higher false-alarm rate of the union is economically justified.

**Practical approach:** Tier the flags rather than using a binary rule. Flag points identified by both methods for immediate, high-priority review. Flag points identified by only one method for a lower-priority queue. This recovers most of the recall benefit (union-level) while managing investigator workload by prioritising the highest-confidence cases — without discarding any signal.

</details>

<details>
<summary>🔑 Reveal summary answers</summary>

1. **DBSCAN noise as anomalies:** Noise points are anomaly *candidates* — they require a second validation step (independent detector or domain context) before being treated as confirmed anomalies, because some sparse-but-normal observations also fall outside cluster boundaries.
2. **Shared judgment (contamination / percentile threshold):** Both require you to specify the expected anomaly proportion in advance. This judgment directly controls the precision/recall tradeoff; error in either direction — overestimating or underestimating the anomaly rate — shifts the detection boundary away from the true optimum.
3. **Method agreement as confidence proxy:** Two independent detectors using fundamentally different mechanisms (density-based vs. isolation-based vs. reconstruction-based) are unlikely to make the same false-alarm error for the same point. When both flag a point, the convergence is strong corroborating evidence of a genuine anomaly — much stronger than either signal alone.

</details>

## Summary

> **Final check:** Answer in one sentence each.

1. What is the main risk of using DBSCAN noise points as anomaly detectors without further validation?
2. `contamination` in Isolation Forest and the percentile threshold in PCA reconstruction error serve analogous roles. What is the shared judgment they both require?
3. Why is method agreement (both methods flagging the same point) a useful proxy for confidence when no ground-truth labels exist?

*(Write your answers here.)*